# InsectVision — cascade training (Colab, free T4 GPU)

Trains both stages of the detection cascade and exports them to single-file
ONNX for the serve-time app (which uses ONNX Runtime only — no torch, no
OpenCV — to fit Render's free tier). Produces three files to download at the
end: `detector.onnx`, `classifier.onnx`, `config/species.json` (updated with
the real accuracy achieved, not a target).

**Before running:** `Runtime -> Change runtime type -> T4 GPU`, then
`Runtime -> Restart session` if you just changed it.

**Two independent halves:**
1. **Classifier (EfficientNet-B0)** — required. Trains on the Kaggle
   folder-per-class dataset you already reviewed in Step 2.
2. **Detector (YOLOv8n)** — optional. Needs a bounding-box dataset from
   Roboflow, which the classifier's data doesn't have. Skip this whole
   section (leave the placeholder values below) and the app runs fine in
   **classifier-only mode** — you just type a count instead of it
   auto-counting. You can always come back and add the detector later.

## 1. GPU check

Fails loudly on purpose — training either model on CPU here would take far too long to be worth starting.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Go to Runtime -> Change runtime type -> "
        "Hardware accelerator -> GPU (T4), then Runtime -> Restart session, "
        "and re-run this notebook from the top."
    )
print(f"GPU OK: {torch.cuda.get_device_name(0)}")

## 1b. Mount Google Drive (recommended)

Detector training below can take 2-4 hours on the free tier, and Colab
sessions *can* disconnect on their own schedule regardless of what you're
doing (inactivity, time limits, or the VM just getting reclaimed).
Ultralytics saves a checkpoint after every epoch -- if that checkpoint only
lives in `/content/` and the session dies, it dies too; on Drive it
doesn't, and training can resume instead of restarting from epoch 0.

Skip this cell (set `USE_DRIVE = False`) if you'd rather not grant Drive
access -- everything still works, you just lose partial progress on a
disconnect during the detector section.

In [ ]:
import os

USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUNS_DIR = "/content/drive/MyDrive/insectvision_runs"
else:
    RUNS_DIR = "/content/runs"

os.makedirs(RUNS_DIR, exist_ok=True)
print(f"Checkpoints will be saved under: {RUNS_DIR}")

## 2. Install dependencies

In [ ]:
# Plain (CPU) onnxruntime on purpose, not onnxruntime-gpu: the export
# verification step below deliberately forces CPUExecutionProvider so it
# matches Render's CPU-only serve environment exactly, so there's no reason
# to pull in the larger CUDA-enabled package just for that check.
!pip install -q ultralytics onnx onnxruntime kaggle roboflow scikit-learn pyyaml

## 3. Upload your reviewed project code

Zip `src/`, `scripts/` and `config/` together from your local `insectvision/`
folder (PowerShell, run from inside `insectvision/`):

```powershell
Compress-Archive -Path src, scripts, config -DestinationPath insectvision_code.zip -Force
```

Upload that zip below. This matters because `config/species.json` already
has the class list and pest/beneficial mapping you reviewed by hand in
Step 2 — re-deriving it here instead would risk silently disagreeing with
what you already checked. Re-running the actual `prepare_data.py` script
(rather than reimplementing its logic in this notebook) keeps the two
environments guaranteed identical.

In [ ]:
from google.colab import files
import json, shutil, sys, zipfile, io
from pathlib import Path

uploaded = files.upload()
print(f"Received {len(uploaded)} file(s): " + ", ".join(f"{n} ({len(b)} bytes)" for n, b in uploaded.items()))
if len(uploaded) != 1:
    raise RuntimeError(
        f"Expected exactly one zip, got {len(uploaded)}. If the file picker "
        "showed a stale filename from an earlier attempt, restart the "
        "runtime (Runtime -> Restart session) and re-run from the top."
    )
zip_name, zip_bytes = next(iter(uploaded.items()))

# Wipe any leftover extraction from a previous (possibly failed) attempt in
# this same session, rather than silently merging old and new files.
target = Path("/content/insectvision")
if target.exists():
    shutil.rmtree(target)

with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
    names = zf.namelist()
    print(f"Zip contains {len(names)} entries. First few:")
    for n in names[:8]:
        print(f"  {n}")
    zf.extractall(target)

sys.path.insert(0, str(target))

species_json = target / "config" / "species.json"
if not species_json.exists():
    found = sorted(str(p.relative_to(target)) for p in target.rglob("*"))
    raise FileNotFoundError(
        f"{species_json} not found after extracting {zip_name}.\n"
        f"Extracted {len(found)} entries instead:\n" + "\n".join(found[:40]) +
        "\n\nMake sure you zipped src/, scripts/ and config/ directly "
        "(so config/species.json is at the zip's root), not the insectvision/ "
        "folder itself (which would nest it one level deeper, e.g. "
        "insectvision/config/species.json)."
    )

species_cfg = json.loads(species_json.read_text())
class_names = species_cfg["class_names"]
taxon_status = species_cfg["taxon_status"]
print(f"\nLoaded {len(class_names)} classes:")
for taxon in class_names:
    print(f"  {taxon:<20} {taxon_status.get(taxon, 'pest')}")


## 4. Kaggle authentication

This CLI version uses a pasted API token, not the classic browser login —
OAuth's browser redirect can't reach back to a headless Colab VM. Get a
token at **kaggle.com/settings/api** ("Create New Token"), paste it below
(hidden input, not saved anywhere but this session).

In [ ]:
import os
from getpass import getpass

kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)
token = getpass("Paste your Kaggle API token: ").strip()
token_path = os.path.join(kaggle_dir, "access_token")
with open(token_path, "w") as f:
    f.write(token)
os.chmod(token_path, 0o600)
print("Kaggle token saved for this session.")

## 5. Download and prepare the classifier dataset

Same command, same script, same defaults as Step 2 locally.

In [ ]:
!kaggle datasets download -d vencerlanz09/agricultural-pests-image-dataset \
    -p /content/data/raw --unzip -q

!python /content/insectvision/scripts/prepare_data.py \
    --source /content/data/raw \
    --out /content/data/classify \
    --species-config /content/insectvision/config/species.json \
    --max-per-class 400 --val-fraction 0.15 --seed 42

species_cfg = json.loads(open("/content/insectvision/config/species.json").read())

## 6. Train the classifier (EfficientNet-B0)

Transfer-learned from ImageNet weights, class-balanced sampling to counter
the dataset's long tail (some taxa have far fewer source images than
others — see the per-class counts Step 2 printed), label smoothing so the
0.75 confidence-referral threshold means something (an overconfident model
would make that threshold toothless). Tracks **macro-F1** (not plain
accuracy) as the model-selection metric, since accuracy alone would be
dominated by whichever classes happen to be most common.

In [ ]:
import torch.nn as nn
from collections import Counter
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, models, transforms
from sklearn.metrics import f1_score, classification_report

device = "cuda"
IMG_SIZE = species_cfg.get("classifier_input_size", 224)

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.4, scale=(0.02, 0.15)),
])
val_tf = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.14)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder("/content/data/classify/train", train_tf)
val_ds = datasets.ImageFolder("/content/data/classify/val", val_tf)

# Class index <-> taxon name must match species.json exactly, or the served
# app would silently report the wrong species for a correct prediction.
# ImageFolder sorts class folder names alphabetically, same as
# prepare_data.py writes class_names -- this assertion catches a mismatch
# immediately instead of it surfacing as a confusing bug at serve time.
assert train_ds.classes == species_cfg["class_names"], (
    f"Class order mismatch.\ndataset:  {train_ds.classes}\n"
    f"species.json: {species_cfg['class_names']}"
)

counts = Counter(train_ds.targets)
weights = [1.0 / counts[t] for t in train_ds.targets]
sampler = WeightedRandomSampler(weights, len(weights), replacement=True)

BATCH = 32
train_dl = DataLoader(train_ds, batch_size=BATCH, sampler=sampler, num_workers=2)
val_dl = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2)

classifier_model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
classifier_model.classifier[1] = nn.Linear(
    classifier_model.classifier[1].in_features, len(train_ds.classes))
classifier_model = classifier_model.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimiser = torch.optim.AdamW(classifier_model.parameters(), lr=3e-4, weight_decay=0.01)

EPOCHS = 30
WARMUP = 3
warmup_sched = torch.optim.lr_scheduler.LinearLR(optimiser, 0.1, 1.0, total_iters=WARMUP)
cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=EPOCHS - WARMUP)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimiser, [warmup_sched, cosine_sched], milestones=[WARMUP])

best_f1, best_state, patience, stale = 0.0, None, 8, 0

for epoch in range(EPOCHS):
    classifier_model.train()
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        optimiser.zero_grad()
        loss = criterion(classifier_model(x), y)
        loss.backward()
        optimiser.step()
    scheduler.step()

    classifier_model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for x, y in val_dl:
            out = classifier_model(x.to(device)).argmax(1).cpu().numpy()
            preds.extend(out.tolist())
            targets.extend(y.numpy().tolist())

    macro_f1 = f1_score(targets, preds, average="macro", zero_division=0)
    print(f"epoch {epoch + 1:3d}/{EPOCHS}  val_macro_f1={macro_f1:.4f}")

    if macro_f1 > best_f1:
        best_f1, stale = macro_f1, 0
        best_state = {k: v.cpu().clone() for k, v in classifier_model.state_dict().items()}
    else:
        stale += 1
        if stale >= patience:
            print(f"early stopping at epoch {epoch + 1}")
            break

classifier_model.load_state_dict(best_state)
classifier_model.eval()

# Re-evaluate the actual best checkpoint (not whichever epoch training
# happened to stop on) so the printed per-class report matches what
# actually gets exported below.
preds, targets = [], []
with torch.no_grad():
    for x, y in val_dl:
        out = classifier_model(x.to(device)).argmax(1).cpu().numpy()
        preds.extend(out.tolist())
        targets.extend(y.numpy().tolist())
classifier_macro_f1 = float(f1_score(targets, preds, average="macro", zero_division=0))

print(f"\nBest checkpoint validation macro-F1: {classifier_macro_f1:.4f}")
print("\nPer-class report:")
print(classification_report(targets, preds, target_names=train_ds.classes, zero_division=0))

## 7. Detector training data (Roboflow) -- already picked for you

Using **FAIR-D v2.5** (workspace `fairdevice-htj8d`, project `fair-d_v2.5`,
version 2): 21,426 images across 39 fine-grained insect-trap classes,
CC BY 4.0. Found via Roboflow Universe search and verified directly
(image/label counts, `data.yaml`) before being wired in below -- not
guessed. Want a different dataset instead? Just edit the three values in
the next cell.

**Why the detector's 39 classes don't need to match the classifier's 12
taxa:** at serve time, the detector's only job is finding *where* insects
are -- species identity always comes from the classifier running on each
cropped region, never from the detector's own class prediction (see
`app.py`, Step 4). So a dataset with a completely different class list
still works fine. By default this notebook collapses every box to one
class, `insect`, before training -- a class-agnostic localiser is an
easier, more data-efficient task, and with 21k+ images merged into a
single class there's plenty of data for it to converge well. Set
`COLLAPSE_TO_SINGLE_CLASS = False` below to keep the original 39 classes
instead, if you'd rather.

**Heads up on training time:** this is a big dataset -- ~1,170 batches/epoch
at batch=16. Expect roughly 3-6 minutes/epoch on a free T4, so the reduced
40-epoch budget below (down from a generic 80) is still likely 2-4 hours.
Free Colab sessions can disconnect on their own schedule regardless of your
activity; if training stops partway, `runs/detect/insectvision_detector/weights/last.pt`
holds the latest checkpoint and `model.train(resume=True)` (pointed at that
run) picks back up rather than restarting from scratch.

In [ ]:
ROBOFLOW_WORKSPACE = "fairdevice-htj8d"  # FAIR-D v2.5 -- 21,426 images, 39 fine classes, CC BY 4.0
ROBOFLOW_PROJECT = "fair-d_v2.5"
ROBOFLOW_VERSION = 2
COLLAPSE_TO_SINGLE_CLASS = True

RUN_DETECTOR_TRAINING = ROBOFLOW_WORKSPACE != "PASTE_WORKSPACE_HERE"  # already filled in below
if not RUN_DETECTOR_TRAINING:
    print("Roboflow project not configured -- skipping detector training.")
    print("The app will run in classifier-only mode. Fill in the three")
    print("values above and re-run this cell onward to add a detector.")
else:
    print(f"Will train detector on {ROBOFLOW_WORKSPACE}/{ROBOFLOW_PROJECT} v{ROBOFLOW_VERSION}")

In [ ]:
if RUN_DETECTOR_TRAINING:
    from getpass import getpass
    from roboflow import Roboflow

    rf_key = getpass("Paste your Roboflow API key (roboflow.com -> Settings -> API key): ").strip()
    rf = Roboflow(api_key=rf_key)
    rf_project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
    rf_dataset = rf_project.version(ROBOFLOW_VERSION).download("yolov8", location="/content/data/detect")
    print(f"Downloaded to {rf_dataset.location}")

In [ ]:
import yaml
from pathlib import Path

DETECT_DIR = Path("/content/data/detect")

if RUN_DETECTOR_TRAINING:
    data_yaml_path = DETECT_DIR / "data.yaml"
    detect_cfg = yaml.safe_load(data_yaml_path.read_text())

    if COLLAPSE_TO_SINGLE_CLASS:
        for split in ("train", "valid", "val", "test"):
            label_dir = DETECT_DIR / split / "labels"
            if not label_dir.exists():
                continue
            for label_file in label_dir.glob("*.txt"):
                fixed = []
                for line in label_file.read_text().splitlines():
                    parts = line.split()
                    if not parts:
                        continue
                    parts[0] = "0"  # collapse every class to a single "insect" class
                    fixed.append(" ".join(parts))
                label_file.write_text("\n".join(fixed) + ("\n" if fixed else ""))
        detect_cfg["names"] = ["insect"]
        detect_cfg["nc"] = 1
        print("Collapsed all boxes to a single 'insect' class.")

    # Rewrite as absolute paths -- Roboflow's exported relative paths are
    # relative to wherever ultralytics happens to resolve them from at
    # train time, which isn't reliably this notebook's cwd.
    val_split = "valid" if (DETECT_DIR / "valid").exists() else "val"
    detect_cfg["path"] = str(DETECT_DIR)
    detect_cfg["train"] = "train/images"
    detect_cfg["val"] = f"{val_split}/images"
    data_yaml_path.write_text(yaml.safe_dump(detect_cfg))
    print(detect_cfg)

## 8. Train the detector (YOLOv8n)

In [ ]:
if RUN_DETECTOR_TRAINING:
    from ultralytics import YOLO

    yolo_model = YOLO("yolov8n.pt")
    yolo_model.train(
        data=str(DETECT_DIR / "data.yaml"),
        epochs=40,  # generous for 21k images; early stopping (patience=20) will cut this short once it plateaus
        imgsz=640,
        batch=16,
        device=0,
        optimizer="SGD",
        lr0=0.01,
        momentum=0.937,
        weight_decay=0.0005,
        warmup_epochs=3,
        cos_lr=True,
        patience=20,
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
        degrees=20.0, scale=0.5, fliplr=0.5, flipud=0.3,
        mosaic=1.0, close_mosaic=10,
        erasing=0.4,
        project=f"{RUNS_DIR}/detect",
        name="insectvision_detector",
    )
    detector_metrics = yolo_model.val()
    detector_map50 = float(detector_metrics.box.map50)
    print(f"\nDetector mAP@0.50 = {detector_map50:.4f}")
else:
    detector_map50 = None

### If you got disconnected during detector training

Re-run the setup cells (GPU check, Drive mount, dependency install, code
upload, Kaggle auth + data prep are all fast) up through the Roboflow
config cell, **then run this cell instead of the training cell above** --
it resumes from the last saved checkpoint rather than starting over. Skip
it entirely if training completed without interruption.

In [ ]:
if RUN_DETECTOR_TRAINING:
    from ultralytics import YOLO

    checkpoint = f"{RUNS_DIR}/detect/insectvision_detector/weights/last.pt"
    yolo_model = YOLO(checkpoint)
    yolo_model.train(resume=True)

    detector_metrics = yolo_model.val()
    detector_map50 = float(detector_metrics.box.map50)
    print(f"\nDetector mAP@0.50 = {detector_map50:.4f}")

## 9. Export both models to single-file ONNX

Newer export paths sometimes switch to external-data storage past a size
threshold (an `.onnx.data` sidecar next to the `.onnx` file), which is easy
to lose track of if only the `.onnx` file gets copied out. `fold_external_data`
collapses everything back into one self-contained file — harmlessly, even
if no sidecar was produced. Each export is then checked against the
original torch model's output on the same input (max drift < 1e-3) before
being trusted.

In [ ]:
import shutil
import numpy as np
import onnx
import onnxruntime as ort

MODELS_DIR = Path("/content/models")
MODELS_DIR.mkdir(exist_ok=True)


def fold_external_data(onnx_path: str) -> None:
    m = onnx.load(onnx_path, load_external_data=True)
    onnx.save(m, onnx_path, save_as_external_data=False)


def assert_close(name: str, torch_out: np.ndarray, onnx_out: np.ndarray, tol: float = 1e-3) -> None:
    drift = float(np.abs(torch_out - onnx_out).max())
    print(f"{name}: torch vs onnx max drift = {drift:.2e} (tolerance {tol:.0e})")
    assert drift < tol, f"{name} ONNX export drifted too far from the torch model ({drift:.2e} >= {tol:.0e})"

In [ ]:
# --- classifier ---
classifier_onnx_path = str(MODELS_DIR / "classifier.onnx")
dummy_classifier_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)

classifier_model.eval().cpu()
with torch.no_grad():
    torch_out = classifier_model(dummy_classifier_input).numpy()

torch.onnx.export(
    classifier_model, dummy_classifier_input, classifier_onnx_path,
    opset_version=17, input_names=["input"], output_names=["output"],
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
)
fold_external_data(classifier_onnx_path)

sess = ort.InferenceSession(classifier_onnx_path, providers=["CPUExecutionProvider"])
onnx_out = sess.run(None, {"input": dummy_classifier_input.numpy()})[0]
assert_close("classifier", torch_out, onnx_out)

classifier_model.to(device)  # move back in case any later cell is re-run

In [ ]:
# --- detector ---
detector_onnx_path = str(MODELS_DIR / "detector.onnx")

if RUN_DETECTOR_TRAINING:
    exported_path = yolo_model.export(format="onnx", imgsz=640, opset=17, simplify=True)
    shutil.copy(exported_path, detector_onnx_path)
    fold_external_data(detector_onnx_path)

    underlying = yolo_model.model.eval().cpu()
    dummy_detector_input = torch.rand(1, 3, 640, 640)
    with torch.no_grad():
        torch_raw = underlying(dummy_detector_input)
        if isinstance(torch_raw, (list, tuple)):
            torch_raw = torch_raw[0]
        torch_raw = torch_raw.numpy()

    sess = ort.InferenceSession(detector_onnx_path, providers=["CPUExecutionProvider"])
    onnx_input_name = sess.get_inputs()[0].name
    onnx_raw = sess.run(None, {onnx_input_name: dummy_detector_input.numpy()})[0]
    assert_close("detector", torch_raw, onnx_raw)
else:
    print("Detector training was skipped (see section 7) -- no detector.onnx produced.")
    print("The app will run in classifier-only mode, which is fully functional.")

## 10. Write real results into species.json

In [ ]:
species_cfg["classifier_macro_f1"] = classifier_macro_f1
species_cfg["detector_map50"] = detector_map50

species_config_out = "/content/insectvision/config/species.json"
with open(species_config_out, "w") as f:
    json.dump(species_cfg, f, indent=2)
    f.write("\n")

print(f"classifier_macro_f1 = {classifier_macro_f1:.4f}")
print(f"detector_map50      = {detector_map50}")
print(f"\nWrote {species_config_out}")

## 11. Download your three files

Place `detector.onnx` and `classifier.onnx` in `insectvision/models/`, and
`species.json` in `insectvision/config/` (overwriting the copy there), then
commit and push — Step 7 covers the actual Render deploy.

In [ ]:
from google.colab import files

files.download(classifier_onnx_path)
if RUN_DETECTOR_TRAINING:
    files.download(detector_onnx_path)
files.download(species_config_out)

## 12. Recovery: re-export a detector from a Drive checkpoint

Use this section if a downloaded `detector.onnx` turns out to have been
exported from the wrong in-memory model (symptom: `detector_map50` comes
back as exactly `0.0`, and the ONNX output shape has 80-something classes
instead of 1 -- that's the untrained base `yolov8n.pt`, not your
fine-tuned checkpoint). This happens if `yolo_model` gets reassigned
between the training cell and the export cell finishing, e.g. from
re-running cells out of order after a disconnect.

**Self-contained** -- doesn't depend on any other cells having run in this
session except "1. GPU check" and "2. Install dependencies" (re-run those
two first if this is a fresh runtime). Loads the checkpoint directly from
Drive rather than trusting whatever `yolo_model` currently holds, and
prints its class count as an explicit sanity check *before* doing anything
else with it.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

CHECKPOINT_PATH = "/content/drive/MyDrive/insectvision_runs/detect/insectvision_detector/weights/best.pt"

from ultralytics import YOLO
yolo_model = YOLO(CHECKPOINT_PATH)
print(f"Loaded checkpoint: {CHECKPOINT_PATH}")
print(f"nc={yolo_model.model.nc}, names={yolo_model.model.names}")

assert yolo_model.model.nc == 1, (
    f"Expected the fine-tuned single-class ('insect') checkpoint but got "
    f"nc={yolo_model.model.nc} ({yolo_model.model.names}) -- this isn't "
    "the right file. Don't continue with this checkpoint."
)
print("\nnc == 1 confirmed -- this is the real fine-tuned checkpoint.")

### Re-download the validation split to measure a real mAP@0.50

Needs your Roboflow API key again (nothing from before was cached).

In [ ]:
from getpass import getpass
from pathlib import Path
import yaml
from roboflow import Roboflow

rf_key = getpass("Paste your Roboflow API key: ").strip()
rf = Roboflow(api_key=rf_key)
rf_dataset = (rf.workspace("fairdevice-htj8d").project("fair-d_v2.5")
             .version(2).download("yolov8", location="/content/data/detect"))

DETECT_DIR = Path("/content/data/detect")
data_yaml_path = DETECT_DIR / "data.yaml"
detect_cfg = yaml.safe_load(data_yaml_path.read_text())

for split in ("train", "valid", "val", "test"):
    label_dir = DETECT_DIR / split / "labels"
    if not label_dir.exists():
        continue
    for label_file in label_dir.glob("*.txt"):
        fixed = []
        for line in label_file.read_text().splitlines():
            parts = line.split()
            if not parts:
                continue
            parts[0] = "0"  # same collapse-to-single-class as the original run
            fixed.append(" ".join(parts))
        label_file.write_text("\n".join(fixed) + ("\n" if fixed else ""))

val_split = "valid" if (DETECT_DIR / "valid").exists() else "val"
detect_cfg["names"] = ["insect"]
detect_cfg["nc"] = 1
detect_cfg["path"] = str(DETECT_DIR)
detect_cfg["train"] = "train/images"
detect_cfg["val"] = f"{val_split}/images"
data_yaml_path.write_text(yaml.safe_dump(detect_cfg))

detector_metrics = yolo_model.val(data=str(data_yaml_path))
detector_map50 = float(detector_metrics.box.map50)
print(f"\nReal detector mAP@0.50 = {detector_map50:.4f}")

### Export + verify (same procedure as section 9, detector only)

In [ ]:
import shutil
import numpy as np
import onnx
import onnxruntime as ort
import torch
from pathlib import Path

MODELS_DIR = Path("/content/models")
MODELS_DIR.mkdir(exist_ok=True)


def fold_external_data(onnx_path: str) -> None:
    m = onnx.load(onnx_path, load_external_data=True)
    onnx.save(m, onnx_path, save_as_external_data=False)


detector_onnx_path = str(MODELS_DIR / "detector.onnx")
exported_path = yolo_model.export(format="onnx", imgsz=640, opset=17, simplify=True)
shutil.copy(exported_path, detector_onnx_path)
fold_external_data(detector_onnx_path)

underlying = yolo_model.model.eval().cpu()
dummy = torch.rand(1, 3, 640, 640)
with torch.no_grad():
    torch_raw = underlying(dummy)
    if isinstance(torch_raw, (list, tuple)):
        torch_raw = torch_raw[0]
    torch_raw = torch_raw.numpy()

sess = ort.InferenceSession(detector_onnx_path, providers=["CPUExecutionProvider"])
onnx_raw = sess.run(None, {sess.get_inputs()[0].name: dummy.numpy()})[0]

print(f"exported output shape: {onnx_raw.shape}  (expect (1, 5, 8400) for a 1-class model)")
assert onnx_raw.shape[1] == 5, f"expected 5 rows (4 box + 1 class), got {onnx_raw.shape[1]} -- still wrong"

drift = float(np.abs(torch_raw - onnx_raw).max())
print(f"torch vs onnx max drift = {drift:.2e} (tolerance 1e-3)")
assert drift < 1e-3, f"drift too high: {drift:.2e}"
print("\nExport verified correct.")

### Download the corrected file

Just `detector.onnx` this time -- your classifier was already correct, no
need to redo it. Overwrite `insectvision/models/detector.onnx` with this,
and update the `detector_map50` value printed above into your local
`config/species.json` by hand (or hand both to me and I'll do it).

In [ ]:
from google.colab import files

files.download(detector_onnx_path)
print(f"\ndetector_map50 = {detector_map50:.4f}  <-- record this in config/species.json")